In [1]:
import os

import pandas as pd

In [2]:
df_contig = pd.read_csv("nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt", sep='\t')

In [6]:
os.makedirs('annotation_out', exist_ok=True)
os.makedirs('IPS_out', exist_ok=True)

final_df = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):
        
        #filter contig list by current file ID
        df_contig_filter = df_contig[df_contig['file'] == f'{file}']
        
        filter_values = set(df_contig_filter['qseqid'])
        
        df_anno = pd.read_csv(f"{project}/annotation_output/features_table/annotations_{file}_scaffolds_1000bp.txt", sep="\t")

        df_anno_filter = df_anno[df_anno['contig_id'].isin(filter_values)]

        df_anno_filter.to_csv(f"annotation_out/annotations_{file}_filter.txt", sep="\t")
        
        filter_values = set(df_anno_filter['feature_id'])
        
        IPS_cols = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O']
        
        df_IPS = pd.read_csv(f"{project}/annotation_output/IPS/{file}_ips.tsv", sep="\t", header = None, names=IPS_cols)

        df_IPS_filter = df_IPS[df_IPS['A'].isin(filter_values)]

        df_IPS_filter.to_csv(f"IPS_out/{file}_IPS_filter.txt", sep="\t")

        df_IPS_filter['file'] = f'{file}'
        
        final_df = pd.concat([final_df, df_IPS_filter], ignore_index=True)
    
final_df.to_csv("IPS_out/IPS_concat.txt", sep='\t', index = False)

/tmp/ipykernel_3950837/970546391.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_IPS_filter['file'] = f'{file}'
/tmp/ipykernel_3950837/970546391.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_IPS_filter['file'] = f'{file}'
/tmp/ipykernel_3950837/970546391.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/

In [3]:
final_df = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):
        df = pd.read_csv(f"IPS_out/{file}_IPS_filter.txt", sep="\t", usecols =['L'])
        df = df.drop_duplicates()
        df['file'] = f'{file}'

        final_df = pd.concat([final_df, df], ignore_index=True)
        
final_df.to_csv("IPS_out/IPS_concat.txt", sep='\t', index = False)


In [5]:
final_df

,L,file
0,IPR002429,250513Pat_D25-7849
1,IPR008972,250513Pat_D25-7849
2,-,250513Pat_D25-7849
3,IPR010514,250513Pat_D25-7849
4,IPR006333,250513Pat_D25-7849
...,...,...
713006,IPR047777,250930Pat_D25-12478
713007,IPR024265,250930Pat_D25-12478
713008,IPR047667,250930Pat_D25-12478
713009,IPR049814,250930Pat_D25-12478


In [ ]:
import pandas as pd
import os
import glob
import string

file_list = glob.glob('annotations_output_folder/*_ips.tsv')
print(file_list)

result_df = pd.DataFrame(columns=['A', 'L', 'file']) #'L' = IPS; 'N' = GO

for file in file_list:
	df = pd.read_csv(f"{file}", header=None, sep ='\t')

	# Generate column names: A, B, C, ...
	df.columns = list(string.ascii_uppercase[:len(df.columns)])

	cols_to_keep = ['A', 'L'] #'L' = IPS; 'N' = GO

	df = df[cols_to_keep]

	df['L'] = df['L'].str.split('|')
	df = df.explode('L', ignore_index=True)
	df = df.drop_duplicates()
	df['file'] = f"{file}"

	result_df = pd.concat([result_df, df], ignore_index=True)

result_df.to_csv('annotations_output_folder/ips_concat.txt', sep = '\t')


In [ ]:
import pandas as pd
import os
import glob
import string

file_list = glob.glob('annotations_output_folder/*_ips.tsv')
print(file_list)

result_df = pd.DataFrame(columns=['A', 'N', 'file']) #'L' = IPS; 'N' = GO

for file in file_list:
	df = pd.read_csv(f"{file}", header=None, sep ='\t')

	# Generate column names: A, B, C, ...
	df.columns = list(string.ascii_uppercase[:len(df.columns)])

	cols_to_keep = ['A', 'N'] #'L' = IPS; 'N' = GO

	df = df[cols_to_keep]

	df['N'] = df['N'].str.split('|')
	df = df.explode('N', ignore_index=True)
	df = df.drop_duplicates()
	df['file'] = f"{file}"

	result_df = pd.concat([result_df, df], ignore_index=True)

result_df.to_csv('annotations_output_folder/go_concat.txt', sep = '\t')

In [ ]:
#!pip install goatools 
import pandas as pd
import os
from goatools import obo_parser

#get go database file --> https://geneontology.org/docs/download-ontology/
#os.system('curl -O https://current.geneontology.org/ontology/go-basic.obo')

# Load the GO terms from the Gene Ontology OBO file
go_obo_file = "go-basic.obo"  # Download from https://github.com/geneontology/go-ontology/blob/master/ontology/go-basic.obo
go_terms = obo_parser.GODag(go_obo_file, load_obsolete=True)


df = pd.read_csv('chi_squared_binary_Year_genus.txt', sep = '\t')
go_id_list = df['GO_term'].tolist()

go_data = []

# Fetch term names by GO ID
for go_id in go_id_list:
    try:
        print(go_id)
        go_term = go_terms[go_id]
        go_data.append({"GO_ID": go_id, "GO_Term_Name": go_term.name})
    except KeyError:
        # If GO ID is not found, append None
        go_data.append({"GO_ID": go_id, "GO_Term_Name": None})
        
# Convert to a Pandas DataFrame
df_go_terms = pd.DataFrame(go_data)

# Display the DataFrame
print(df_go_terms)


dfm = pd.merge(df, df_go_terms, left_on = 'GO_term', right_on = 'GO_ID', how = 'left')

dfm.to_csv("chi_squared_binary_Year_genus_go_names.txt", sep = '\t', index = False)